# Étape 3 : Data Preparation (Préparation des Données)

Ce notebook implémente le pipeline complet de préparation des données pour nos 5 objectifs de modélisation, incluant le nettoyage, l'imputation, l'encodage et la mise à l'échelle.

In [11]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

# Chargement du dataset
df = pd.read_csv('f1_modeling_ready.csv')

### Step 1 — Drop leakage columns

Suppression des colonnes qui contiennent des informations post-course.

In [12]:
leakage_cols = ['positionOrder', 'points', 'laps', 'rank', 'statusId']
df_clean = df.drop(columns=leakage_cols)

print(f"Colonnes conservées : {df_clean.columns.tolist()}")

Colonnes conservées : ['year', 'round', 'grid', 'qualifying_position', 'driver_standing_pos', 'driver_points_cum', 'driver_wins_cum', 'constructor_standing_pos', 'constructor_points_cum', 'constructor_wins_cum', 'country', 'podium', 'position_class', 'finish_pos_penalty']


### Step 2 — Handle missing values

Imputation des positions au classement par la médiane groupée par année.

In [3]:
print("Valeurs manquantes avant imputation :")
print(df_clean.isnull().sum())

# Imputation driver_standing_pos
df_clean['driver_standing_pos'] = df_clean.groupby('year')['driver_standing_pos'].transform(lambda x: x.fillna(x.median()))

# Imputation constructor_standing_pos
df_clean['constructor_standing_pos'] = df_clean.groupby('year')['constructor_standing_pos'].transform(lambda x: x.fillna(x.median()))

# Suppression des lignes restantes avec des valeurs manquantes
df_clean = df_clean.dropna()

print("\nValeurs manquantes après imputation et nettoyage :")
print(df_clean.isnull().sum())

Valeurs manquantes avant imputation :
year                            0
round                           0
grid                            0
qualifying_position         16265
driver_standing_pos           469
driver_points_cum             469
driver_wins_cum               469
constructor_standing_pos     1867
constructor_points_cum       1867
constructor_wins_cum         1867
country                         0
podium                          0
position_class                  0
finish_pos_penalty              0
dtype: int64

Valeurs manquantes après imputation et nettoyage :
year                        0
round                       0
grid                        0
qualifying_position         0
driver_standing_pos         0
driver_points_cum           0
driver_wins_cum             0
constructor_standing_pos    0
constructor_points_cum      0
constructor_wins_cum        0
country                     0
podium                      0
position_class              0
finish_pos_penalty          0
d

### Step 3 — Encode country column

Encodage des pays et des classes de position.

In [13]:
le_country = LabelEncoder()
df_clean['country_encoded'] = le_country.fit_transform(df_clean['country'])

le_pos_class = LabelEncoder()
df_clean['position_class_encoded'] = le_pos_class.fit_transform(df_clean['position_class'])

### Step 4 — Define feature sets

In [14]:
FEATURES_WITH_COUNTRY = [
    'year', 'round', 'grid', 'qualifying_position', 
    'driver_standing_pos', 'driver_points_cum', 'driver_wins_cum', 
    'constructor_standing_pos', 'constructor_points_cum', 
    'constructor_wins_cum', 'country_encoded'
]

FEATURES_NO_COUNTRY = [
    'year', 'round', 'grid', 'qualifying_position', 
    'driver_standing_pos', 'driver_points_cum', 'driver_wins_cum', 
    'constructor_standing_pos', 'constructor_points_cum', 
    'constructor_wins_cum'
]

### Step 5 — Create (X, y) for each objective

In [15]:
# OBJ1: Binary classification - podium
X1 = df_clean[FEATURES_WITH_COUNTRY]
y1 = df_clean['podium']

# OBJ2: Multiclass classification - position_class
X2 = df_clean[FEATURES_WITH_COUNTRY]
y2 = df_clean['position_class_encoded']

# OBJ3: Regression - finish_pos_penalty (with country)
X3 = df_clean[FEATURES_WITH_COUNTRY]
y3 = df_clean['finish_pos_penalty']

# OBJ4: Clustering (features only)
X4 = df_clean[FEATURES_WITH_COUNTRY]

# OBJ5: Regression - finish_pos_penalty (no country)
X5 = df_clean[FEATURES_NO_COUNTRY]
y5 = df_clean['finish_pos_penalty']

### Step 6 & 7 — Train/Test Split & Scaling

In [16]:
def split_and_scale(X, y, stratify=False):
    strat = y if stratify else None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=strat)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

# OBJ1 split & scale
X1_train, X1_test, y1_train, y1_test, sc1 = split_and_scale(X1, y1, stratify=True)

# OBJ2 split & scale
X2_train, X2_test, y2_train, y2_test, sc2 = split_and_scale(X2, y2, stratify=True)

# OBJ3 split & scale
X3_train, X3_test, y3_train, y3_test, sc3 = split_and_scale(X3, y3, stratify=False)

# OBJ4 (Clustering) - Scale only
sc4 = StandardScaler()
X4_scaled = sc4.fit_transform(X4)

# OBJ5 split & scale
X5_train, X5_test, y5_train, y5_test, sc5 = split_and_scale(X5, y5, stratify=False)

### Step 8 — SMOTE on OBJ1 train set only

In [17]:
smote = SMOTE(random_state=42)
X1_train_res, y1_train_res = smote.fit_resample(X1_train, y1_train)

print(f"OBJ1 avant SMOTE: {np.bincount(y1_train)}")
print(f"OBJ1 après SMOTE: {np.bincount(y1_train_res)}")

ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

### Step 9 — Save everything

Sauvegarde des objets binaires (PKL) pour les modèles et du dataset nettoyé (CSV).

In [18]:
data_to_save = {
    'OBJ1': (X1_train_res, X1_test, y1_train_res, y1_test, sc1),
    'OBJ2': (X2_train, X2_test, y2_train, y2_test, sc2),
    'OBJ3': (X3_train, X3_test, y3_train, y3_test, sc3),
    'OBJ4': (X4_scaled, sc4),
    'OBJ5': (X5_train, X5_test, y5_train, y5_test, sc5),
    'encoders': {'country': le_country, 'pos_class': le_pos_class}
}

# Sauvegarde PKL
with open('prepared_data.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)

# Exportation du dataset nettoyé en CSV
df_clean.to_csv('prepared_data.csv', index=False)

print("Données préparées sauvegardées avec succès (prepared_data.pkl et prepared_data.csv).")

Données préparées sauvegardées avec succès (prepared_data.pkl et prepared_data.csv).


### Final Summary

In [19]:
print("Résumé des dimensions (Train / Test) :")
print(f"OBJ1 : Train {X1_train_res.shape}, Test {X1_test.shape}")
print(f"OBJ2 : Train {X2_train.shape}, Test {X2_test.shape}")
print(f"OBJ3 : Train {X3_train.shape}, Test {X3_test.shape}")
print(f"OBJ5 : Train {X5_train.shape}, Test {X5_test.shape}")

Résumé des dimensions (Train / Test) :
OBJ1 : Train (13676, 11), Test (5300, 11)
OBJ2 : Train (21199, 11), Test (5300, 11)
OBJ3 : Train (21199, 11), Test (5300, 11)
OBJ5 : Train (21199, 10), Test (5300, 10)
